# Step 2 — Exploratory Data Analysis: IBM AML (HI-Small)

**Runs on Kaggle** (not the laptop). Add the dataset *IBM Transactions for Anti Money Laundering (AML)* as input, then **Run All**.

Goals:
1. Know the exact size, date range and class imbalance.
2. Check the daily laundering rate (decides how we do the time split).
3. Understand which attributes relate to laundering.
4. **Risk check:** does the patterns file (fan-in, cycle, ...) map onto transactions?

Output: `eda_summary.md` + figures in `/kaggle/working/`.

In [ ]:
import os, glob, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)

# Find the data: Kaggle input folder, or a local folder via AML_DATA_DIR (used for testing).
search_root = os.environ.get("AML_DATA_DIR", "/kaggle/input")
trans_files = glob.glob(f"{search_root}/**/HI-Small_Trans.csv", recursive=True)
pattern_files = glob.glob(f"{search_root}/**/HI-Small_Patterns.txt", recursive=True)
assert trans_files, f"HI-Small_Trans.csv not found under {search_root}. Did you add the dataset as input?"
TRANS_PATH = trans_files[0]
PATTERNS_PATH = pattern_files[0] if pattern_files else None
OUT = Path(os.environ.get("AML_OUT_DIR", "/kaggle/working"))
OUT.mkdir(parents=True, exist_ok=True)
print("transactions:", TRANS_PATH)
print("patterns:    ", PATTERNS_PATH)
summary = []          # lines for eda_summary.md
def note(line=""):
    print(line); summary.append(line)

## 1. Load with memory-efficient types

In [ ]:
t0 = time.time()
raw = pd.read_csv(
    TRANS_PATH,
    dtype={"From Bank": "int32", "Account": "string", "To Bank": "int32", "Account.1": "string",
           "Amount Received": "float64", "Receiving Currency": "category",
           "Amount Paid": "float64", "Payment Currency": "category",
           "Payment Format": "category", "Is Laundering": "int64"},  # int64: int8 would overflow in 100*count,
)
# Standard column names (same mapping as configs/schema.yaml)
df = raw.rename(columns={
    "Timestamp": "timestamp", "From Bank": "from_bank", "Account": "from_account",
    "To Bank": "to_bank", "Account.1": "to_account", "Amount Received": "amount_received",
    "Receiving Currency": "receiving_currency", "Amount Paid": "amount_paid",
    "Payment Currency": "payment_currency", "Payment Format": "payment_format",
    "Is Laundering": "label"})
del raw
df["timestamp"] = pd.to_datetime(df["timestamp"], format="%Y/%m/%d %H:%M")
df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)
print(f"loaded in {time.time()-t0:.1f}s, memory {df.memory_usage(deep=True).sum()/1e9:.2f} GB")
df.head()

## 2. Size, date range, class imbalance

In [ ]:
n = len(df); n_pos = int(df["label"].sum())
note("# EDA summary — IBM AML HI-Small")
note()
note("## Size and imbalance")
note(f"- Transactions: {n:,}")
note(f"- Laundering transactions: {n_pos:,} ({n_pos/n:.4%})")
note(f"- Ratio normal:laundering ≈ {(n-n_pos)/max(n_pos,1):,.0f}:1")
note(f"- Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")
accounts = pd.unique(pd.concat([df["from_account"], df["to_account"]]))
note(f"- Unique accounts: {len(accounts):,}")
note(f"- Unique banks: {pd.unique(pd.concat([df['from_bank'], df['to_bank']])).size:,}")
note(f"- Currencies: {df['payment_currency'].nunique()} | Payment formats: {df['payment_format'].nunique()}")

## 3. Daily volume and laundering rate (decides the time split)

In [ ]:
daily = df.groupby(df["timestamp"].dt.date).agg(n=("label", "size"), pos=("label", "sum"))
daily["rate_%"] = 100 * daily["pos"] / daily["n"]
daily["cum_share_%"] = 100 * daily["n"].cumsum() / daily["n"].sum()
print(daily.to_string())
note()
note("## Daily volume")
note(f"- Days: {len(daily)}; median transactions/day: {int(daily['n'].median()):,}")
tail = daily[daily["n"] < 0.05 * daily["n"].median()]
if len(tail):
    note(f"- Sparse tail days (<5% of median volume): {len(tail)} days from {tail.index.min()}, "
         f"{int(tail['n'].sum()):,} txns, laundering rate {100*tail['pos'].sum()/tail['n'].sum():.2f}%")
    note("  → These days are unrepresentative; the time split should be based on the dense period.")
else:
    note("- No sparse tail days found.")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
daily["n"].plot(ax=ax[0], marker="o", title="Transactions per day"); ax[0].set_yscale("log")
daily["rate_%"].plot(ax=ax[1], marker="o", color="crimson", title="Laundering rate per day (%)")
plt.tight_layout(); plt.savefig(OUT / "fig_daily.png", dpi=110); plt.show()

## 4. Which attributes relate to laundering?

In [ ]:
def rate_table(col):
    t = df.groupby(col, observed=True)["label"].agg(n="size", pos="sum")
    t["rate_%"] = 100 * t["pos"] / t["n"]
    t["lift"] = (t["pos"] / t["n"]) / (n_pos / n)
    return t.sort_values("rate_%", ascending=False)

base = 100 * n_pos / n
note()
note("## Laundering rate by attribute (overall rate = %.3f%%)" % base)
for col in ["payment_format", "payment_currency"]:
    t = rate_table(col)
    print(f"\n--- {col} ---\n", t.to_string())
    note(f"- **{col}** top: " + ", ".join(f"{i} {r['rate_%']:.3f}% (lift {r['lift']:.1f})"
                                       for i, r in t.head(3).iterrows()))

df["hour"] = df["timestamp"].dt.hour
h = rate_table("hour").sort_index()
note(f"- **hour of day**: highest rate at hour {int(h['rate_%'].idxmax())} ({h['rate_%'].max():.3f}%), "
     f"lowest at hour {int(h['rate_%'].idxmin())} ({h['rate_%'].min():.3f}%)")

df["cross_currency"] = (df["payment_currency"].astype(str) != df["receiving_currency"].astype(str)).astype("int8")
df["self_transfer"] = (df["from_account"] == df["to_account"]).astype("int8")
df["cross_bank"] = (df["from_bank"] != df["to_bank"]).astype("int8")
for col in ["cross_currency", "self_transfer", "cross_bank"]:
    t = rate_table(col)
    note(f"- **{col}**: " + ", ".join(f"{i}: {r['rate_%']:.3f}% of {int(r['n']):,}" for i, r in t.sort_index().iterrows()))

In [ ]:
# Amounts differ by currency, so compare within the most common currency only.
top_ccy = df["payment_currency"].value_counts().index[0]
sub = df[df["payment_currency"] == top_ccy]
q = sub.groupby("label")["amount_paid"].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).unstack()
print(f"Amount paid quantiles ({top_ccy}) by label:\n", q.round(2).to_string())
note(f"- **amount ({top_ccy})** median: normal {q.loc[0, 0.5]:,.2f} vs laundering {q.loc[1, 0.5]:,.2f}" if 1 in q.index else "")

fig, ax = plt.subplots(figsize=(7, 4))
for lab, c in [(0, "steelblue"), (1, "crimson")]:
    v = np.log10(sub.loc[sub["label"] == lab, "amount_paid"].clip(lower=0.01))
    ax.hist(v, bins=60, density=True, alpha=0.5, color=c, label=["normal", "laundering"][lab])
ax.set_xlabel(f"log10(amount paid, {top_ccy})"); ax.legend(); ax.set_title("Amount distribution by class")
plt.tight_layout(); plt.savefig(OUT / "fig_amounts.png", dpi=110); plt.show()

## 5. Account-level view (motivates fan-in / fan-out features)

In [ ]:
out_deg = df.groupby("from_account")["to_account"].nunique()
in_deg = df.groupby("to_account")["from_account"].nunique()
laund_senders = set(df.loc[df["label"] == 1, "from_account"])
laund_receivers = set(df.loc[df["label"] == 1, "to_account"])
note()
note("## Accounts")
note(f"- Accounts that sent ≥1 laundering txn: {len(laund_senders):,}; received ≥1: {len(laund_receivers):,}")
note(f"- Median unique recipients — laundering senders: {out_deg[out_deg.index.isin(laund_senders)].median():.0f} "
     f"vs others: {out_deg[~out_deg.index.isin(laund_senders)].median():.0f}")
note(f"- Median unique senders — laundering receivers: {in_deg[in_deg.index.isin(laund_receivers)].median():.0f} "
     f"vs others: {in_deg[~in_deg.index.isin(laund_receivers)].median():.0f}")

## 6. Risk check — do the laundering patterns map onto transactions?

In [ ]:
note()
note("## Patterns file check")
if PATTERNS_PATH is None:
    note("- Patterns file not found → per-pattern error analysis will use segments instead.")
else:
    rows, ptype = [], None
    with open(PATTERNS_PATH, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("BEGIN LAUNDERING ATTEMPT"):
                ptype = line.split("-", 1)[1].strip().split(":")[0].strip()
            elif line.startswith("END LAUNDERING ATTEMPT"):
                ptype = None
            elif ptype is not None:
                p = line.split(",")
                rows.append((p[0], int(p[1]), p[2], int(p[3]), p[4], round(float(p[7]), 2), ptype))
    pat = pd.DataFrame(rows, columns=["timestamp", "from_bank", "from_account", "to_bank",
                                      "to_account", "amount_paid", "pattern"])
    pat["timestamp"] = pd.to_datetime(pat["timestamp"], format="%Y/%m/%d %H:%M")
    note(f"- Pattern transactions listed: {len(pat):,}")
    for k, v in pat["pattern"].value_counts().items():
        note(f"  - {k}: {v:,}")

    key = ["timestamp", "from_bank", "from_account", "to_bank", "to_account", "amount_paid"]
    pos = df.loc[df["label"] == 1, key].copy()
    pos["amount_paid"] = pos["amount_paid"].round(2)
    pat_keys = pat.drop_duplicates(key)
    m = pos.merge(pat_keys, on=key, how="left")
    matched = m["pattern"].notna().mean()
    note(f"- Laundering transactions matched to a pattern: {matched:.1%}")
    if matched >= 0.5:
        note("  → OK: per-pattern recall analysis is possible in Step 7.")
    else:
        note("  → Low match: Step 7 will analyse errors by segment (format, currency, amount) instead.")
    pat.to_csv(OUT / "pattern_transactions.csv", index=False)

## 7. Save the summary

In [ ]:
(OUT / "eda_summary.md").write_text("\n".join(summary), encoding="utf-8")
daily.to_csv(OUT / "daily_counts.csv")
print("Saved:", sorted(p.name for p in OUT.iterdir()))